In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning --quiet

In [ ]:
"""TFT v2 백테스트
TFT v2 예측을 기반으로 포트폴리오 시뮬레이션.
기존 PortfolioSimulator와 동일한 전략:
  - Top-10 종목 매수
  - 매도: 손절 -3% / 모델 < 0.45 / 익절 +7% / 5일 타임아웃
  - 수수료: 토스증권 (매수 0.015%, 매도 0.215%)
"""
import warnings, json
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
RAW_OHLCV_PATH = BASE_PATH / "raw" / "ohlcv"
RAW_MACRO_PATH = BASE_PATH / "raw" / "macro"
MODEL_SAVE_PATH = BASE_PATH / "models" / "tft_v2_backtest"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TRAIN_END = "2024-12-31"
BT_START = "2025-01-01"
BT_END = "2026-01-31"
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# 백테스트 전략 설정
INITIAL_CAPITAL = 100_000_000
BUY_COMMISSION = 0.00015
SELL_COMMISSION = 0.00215
TOP_N = 10
STOP_LOSS = -0.03
TAKE_PROFIT = 0.07
MODEL_EXIT = 0.45
MAX_HOLD_DAYS = 5
MAX_POSITIONS = 20

print(f"백테스트: {BT_START} ~ {BT_END}")
print(f"전략: Top-{TOP_N}, 손절 {STOP_LOSS*100}%, 익절 +{TAKE_PROFIT*100}%, {MAX_HOLD_DAYS}일 타임아웃")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"TFT 피처: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): x,y=b; l=self.loss_fn(self(x),y); self.log("train_loss",l,prog_bar=True); return l
    def validation_step(self,b,_):
        x,y=b; lo=self(x); self.log("val_loss",self.loss_fn(lo,y),prog_bar=True)
    def configure_optimizers(self):
        o=torch.optim.AdamW(self.parameters(),lr=5e-4,weight_decay=1e-4)
        s=torch.optim.lr_scheduler.ReduceLROnPlateau(o,"min",0.5,patience=5,min_lr=1e-6)
        return {"optimizer":o,"lr_scheduler":{"scheduler":s,"monitor":"val_loss"}}

def make_samples(full_df, start, end, seq_len, feats):
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

# ===== 체크포인트 로드 or 신규 학습 =====
CKPT_PATH = MODEL_SAVE_PATH / "best.ckpt"
# 다른 경로의 체크포인트도 탐색
ALT_CKPT_PATHS = [
    BASE_PATH / "models" / "tft_v2_backtest" / "best.ckpt",
    BASE_PATH / "models" / "tft_v2" / "best.ckpt",
]

ckpt_found = None
for p in [CKPT_PATH] + ALT_CKPT_PATHS:
    if p.exists():
        ckpt_found = p
        break

if ckpt_found is not None:
    print(f"체크포인트 로드: {ckpt_found}")
    best = TFTv2.load_from_checkpoint(str(ckpt_found), strict=False)
    best.eval(); best.cuda()
    print("로드 완료 (학습 스킵)")
else:
    print("체크포인트 없음 → 신규 학습 시작...")
    train_samples, _ = make_samples(df, "2008-01-01", TRAIN_END, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    labels = [s[1] for s in train_samples]
    n0=sum(1 for l in labels if l==0); n1=sum(1 for l in labels if l==1)
    cw = torch.tensor([len(labels)/(2*max(n0,1)), len(labels)/(2*max(n1,1))], dtype=torch.float32)

    model = TFTv2(N_FEATURES, MAX_ENCODER_LENGTH, 128, 4, 1, 0.2, 2, 5e-4, cw)
    val_size = max(len(train_samples)//10, 1000)
    trainer = pl.Trainer(max_epochs=50, accelerator="gpu", devices=1, gradient_clip_val=0.1,
        callbacks=[EarlyStopping(monitor="val_loss",patience=8,mode="min",verbose=False)],
        enable_progress_bar=True, log_every_n_steps=50)
    trainer.fit(model,
        train_dataloaders=DataLoader(SeqDS(train_samples[:-val_size]),batch_size=BATCH_SIZE,shuffle=True,num_workers=0),
        val_dataloaders=DataLoader(SeqDS(train_samples[-val_size:]),batch_size=BATCH_SIZE*2,shuffle=False,num_workers=0))

    best = TFTv2.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, strict=False)
    best.eval(); best.cuda()
    # 체크포인트 저장
    trainer.save_checkpoint(str(CKPT_PATH))
    print(f"학습 완료 & 체크포인트 저장: {CKPT_PATH}")

In [ ]:
# ===== 백테스트 기간 예측 생성 =====
bt_samples, bt_metas = make_samples(df, BT_START, BT_END, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
bt_loader = DataLoader(SeqDS(bt_samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

bt_probs = []
with torch.no_grad():
    for x, y in bt_loader:
        p = torch.softmax(best(x.cuda()), dim=-1)[:, 1].cpu().numpy()
        bt_probs.extend(p)
bt_probs = np.array(bt_probs)

# 날짜별 예측 dict: {date: {ticker: prob}}
predictions = {}
for meta, prob in zip(bt_metas, bt_probs):
    d, t = meta["date"], meta["ticker"]
    if d not in predictions: predictions[d] = {}
    predictions[d][t] = float(prob)

print(f"예측 완료: {len(predictions)} 거래일, {len(bt_probs):,} 예측")

In [ ]:
# ===== 종가 데이터 로드 =====
tickers_needed = set()
for dp in predictions.values(): tickers_needed.update(dp.keys())

close_prices = {}  # {ticker: Series(date->close)}
for ticker in tickers_needed:
    path = RAW_OHLCV_PATH / f"{ticker}.parquet"
    if not path.exists(): continue
    try:
        ohlcv = pd.read_parquet(str(path))
        ohlcv.index = pd.to_datetime(ohlcv.index)
        close_prices[ticker] = ohlcv["close"].sort_index()
    except: pass

print(f"종가 로드: {len(close_prices)} 종목")

# KOSPI200 벤치마크
kospi_path = RAW_MACRO_PATH / "kospi200.parquet"
kospi = pd.Series(dtype=float)
if kospi_path.exists():
    kdf = pd.read_parquet(str(kospi_path))
    kdf.index = pd.to_datetime(kdf.index)
    col = "close" if "close" in kdf.columns else kdf.columns[0]
    kospi = kdf[col].sort_index()
    print(f"KOSPI200 로드: {len(kospi)} rows")

In [ ]:
# ===== 포트폴리오 시뮬레이터 =====
@dataclass
class Position:
    ticker: str; buy_date: str; buy_price: float; shares: int
    buy_prob: float; invested: float; hold_days: int = 0

def get_close(ticker, date):
    if ticker not in close_prices: return None
    s = close_prices[ticker]; ts = pd.Timestamp(date)
    if ts in s.index: return float(s.loc[ts])
    mask = s.index <= ts
    return float(s.loc[mask].iloc[-1]) if mask.any() else None

# 시뮬레이션
cash = INITIAL_CAPITAL
positions = []
trade_log = []
snapshots = []

trading_dates = sorted(predictions.keys())
print(f"시뮬레이션: {len(trading_dates)} 거래일")

for i, date in enumerate(trading_dates):
    # 보유일 증가
    for p in positions: p.hold_days += 1
    
    # 매도 체크
    date_preds = predictions.get(date, {})
    sells = []
    for pos in positions:
        price = get_close(pos.ticker, date)
        if price is None: continue
        ret = (price - pos.buy_price) / pos.buy_price
        if ret <= STOP_LOSS:
            sells.append((pos, f"stop_loss ({ret*100:.1f}%)")); continue
        cp = date_preds.get(pos.ticker)
        if cp is not None and cp < MODEL_EXIT:
            sells.append((pos, f"model_exit (p={cp:.3f})")); continue
        if ret >= TAKE_PROFIT:
            sells.append((pos, f"take_profit ({ret*100:.1f}%)")); continue
        if pos.hold_days >= MAX_HOLD_DAYS:
            sells.append((pos, f"timeout ({pos.hold_days}d)"))
    
    for pos, reason in sells:
        price = get_close(pos.ticker, date)
        if price is None: continue
        rev = pos.shares * price
        comm = rev * SELL_COMMISSION
        net = rev - comm
        pnl = net - pos.invested
        cash += net
        positions.remove(pos)
        trade_log.append({"date":date,"ticker":pos.ticker,"action":"SELL","price":price,
            "shares":pos.shares,"amount":net,"commission":comm,"pnl":pnl,
            "return":pnl/pos.invested,"hold_days":pos.hold_days,"reason":reason})
    
    # 매수
    preds = predictions[date]
    held = {p.ticker for p in positions}
    candidates = [(t,p) for t,p in preds.items() if t not in held and p > 0.5]
    candidates.sort(key=lambda x: -x[1])
    buys = candidates[:TOP_N]
    
    for ticker, prob in buys:
        if len(positions) >= MAX_POSITIONS: break
        price = get_close(ticker, date)
        if price is None or price <= 0: continue
        slots = MAX_POSITIONS - len(positions)
        alloc = min(cash / max(slots, 1), cash)
        shares = int(alloc / (price * (1 + BUY_COMMISSION)))
        if shares <= 0: continue
        cost = shares * price; comm = cost * BUY_COMMISSION; total = cost + comm
        if total > cash: continue
        cash -= total
        positions.append(Position(ticker, date, price, shares, prob, total))
        trade_log.append({"date":date,"ticker":ticker,"action":"BUY","price":price,
            "shares":shares,"amount":total,"commission":comm,"prob":prob})
    
    # 스냅샷
    pv = cash + sum(p.shares * (get_close(p.ticker, date) or 0) for p in positions)
    snapshots.append({"date":date,"portfolio_value":pv,"cash":cash,"n_positions":len(positions)})
    
    if (i+1) % 50 == 0:
        print(f"  Day {i+1}/{len(trading_dates)} | {date} | 자산: {pv:,.0f} | 수익: {(pv/INITIAL_CAPITAL-1)*100:+.2f}%")

# 잔여 포지션 청산
if positions:
    last = trading_dates[-1]
    print(f"잔여 {len(positions)} 포지션 청산")
    for pos in list(positions):
        price = get_close(pos.ticker, last)
        if price is None: continue
        rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
        cash += net; positions.remove(pos)
        trade_log.append({"date":last,"ticker":pos.ticker,"action":"SELL","price":price,
            "shares":pos.shares,"amount":net,"commission":comm,"pnl":net-pos.invested,
            "return":(net-pos.invested)/pos.invested,"hold_days":pos.hold_days,"reason":"backtest_end"})

print(f"\n시뮬레이션 완료")

In [ ]:
# ===== 결과 계산 =====
snap_df = pd.DataFrame(snapshots)
snap_df["date"] = pd.to_datetime(snap_df["date"])
trade_df = pd.DataFrame(trade_log)
sell_df = trade_df[trade_df["action"]=="SELL"] if len(trade_df) > 0 else pd.DataFrame()

final_value = snap_df["portfolio_value"].iloc[-1]
total_return = final_value / INITIAL_CAPITAL - 1
n_days = len(snap_df)
annual_return = (1 + total_return) ** (252 / max(n_days, 1)) - 1

daily_ret = snap_df["portfolio_value"].pct_change().dropna()
sharpe = float(daily_ret.mean() / daily_ret.std() * np.sqrt(252)) if daily_ret.std() > 1e-9 else 0
cummax = snap_df["portfolio_value"].cummax()
mdd = float(((snap_df["portfolio_value"] - cummax) / cummax).min())

if len(sell_df) > 0:
    win_rate = (sell_df["pnl"] > 0).mean()
    total_comm = trade_df["commission"].sum()
    avg_win = sell_df[sell_df["pnl"]>0]["return"].mean() if (sell_df["pnl"]>0).any() else 0
    avg_loss = sell_df[sell_df["pnl"]<=0]["return"].mean() if (sell_df["pnl"]<=0).any() else 0
else:
    win_rate = total_comm = avg_win = avg_loss = 0

# 벤치마크
bm_ret = None
bt_s, bt_e = pd.Timestamp(BT_START), snap_df["date"].iloc[-1]
bm_slice = kospi[(kospi.index >= bt_s) & (kospi.index <= bt_e)]
if len(bm_slice) >= 2:
    bm_ret = float(bm_slice.iloc[-1] / bm_slice.iloc[0] - 1)

print("="*70)
print("  TFT v2 포트폴리오 백테스트 결과")
print("="*70)
print(f"  기간: {BT_START} ~ {str(snap_df['date'].iloc[-1].date())} ({n_days}일)")
print(f"  초기 자본: {INITIAL_CAPITAL:>15,}원")
print(f"  최종 자산: {final_value:>15,.0f}원")
print(f"  총 수익률: {total_return*100:>+8.2f}%")
print(f"  연환산:    {annual_return*100:>+8.2f}%")
print(f"  Sharpe:    {sharpe:>8.3f}")
print(f"  MDD:       {mdd*100:>8.2f}%")
if bm_ret is not None:
    print(f"  KOSPI200:  {bm_ret*100:>+8.2f}%")
    print(f"  초과수익:  {(total_return-bm_ret)*100:>+8.2f}%")
print(f"  총 거래:   {len(sell_df):>8}건")
print(f"  승률:      {win_rate*100:>8.1f}%")
print(f"  평균이익:  {avg_win*100:>+8.2f}%")
print(f"  평균손실:  {avg_loss*100:>+8.2f}%")
print(f"  총수수료:  {total_comm:>12,.0f}원")
print()
if len(sell_df) > 0:
    reasons = sell_df["reason"].apply(lambda x: x.split(" ")[0]).value_counts()
    print("  [매도 사유]")
    for r, c in reasons.items():
        print(f"    {r:16s}: {c:5d}건 ({c/len(sell_df)*100:.1f}%)")
print("="*70)

# 비교
print(f"\n비교: LSTM Top-10 수익률=+48.95%, Sharpe=1.974, MDD=-17.75%")

In [ ]:
# ===== 저장 =====
snap_df.to_parquet(str(MODEL_SAVE_PATH / f"equity_curve_{datetime.now().strftime('%Y%m%d')}.parquet"))
if len(trade_df) > 0:
    trade_df.to_parquet(str(MODEL_SAVE_PATH / f"trade_log_{datetime.now().strftime('%Y%m%d')}.parquet"))

summary = {"model": "TFT_v2", "period": f"{BT_START}~{BT_END}",
  "total_return": round(total_return*100, 2), "annual_return": round(annual_return*100, 2),
  "sharpe": round(sharpe, 3), "mdd": round(mdd*100, 2),
  "win_rate": round(win_rate*100, 1), "total_trades": len(sell_df),
  "total_commission": round(total_comm), "benchmark": round(bm_ret*100,2) if bm_ret else None}
json.dump(summary, open(str(MODEL_SAVE_PATH/"backtest_summary.json"),"w"), indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## TFT v2 백테스트\n\n### 전략\n- 매수: TFT v2 상승확률 Top-10\n- 매도: 손절 -3% / 모델 < 0.45 / 익절 +7% / 5일 타임아웃\n- 수수료: 토스증권 (왕복 0.23%)\n\n### 비교 기준\n| 모델 | 수익률 | Sharpe | MDD |\n|------|--------|--------|-----|\n| LSTM Top-10 | +48.95% | 1.974 | -17.75% |\n| **TFT v2 Top-10** | **?%** | **?** | **?%** |